# HERMES on Google Colab

**Hierarchical Entropy Routing Model with Efficient State-spaces** — a neural
adaptive byte compressor. Metric: **BPC (bits per byte)**, lower is better.

This notebook is the Colab counterpart of the project's `kaggle_cell.py`.

### Before you run
1. **Runtime → Change runtime type → Hardware accelerator: GPU** (T4 is enough;
   Colab Pro L4 / V100 / A100 also work).
2. *(Optional)* **Secrets** (🔑 icon, left sidebar): add one named `HF` with a
   HuggingFace token and enable *Notebook access* for faster/gated datasets.
3. *(Optional)* Keep `USE_DRIVE = True` (next cell) so checkpoints land in Google
   Drive and survive Colab's idle / 12-hour disconnects — `resume=True` then
   continues a later session from where it stopped.

> **Time on a T4:** `p1=10, p2=15` ≈ 9 h · full `p1=15, p2=25` ≈ 13–14 h ·
> each benchmark/finetune iteration ≈ 30–60 min.

## 1 · Setup — clone, install, (optional) Drive + HF auth

In [ ]:
import os, sys, subprocess

USE_DRIVE = True   # mount Drive for persistent checkpoints + cross-session resume
GIT_URL   = 'https://github.com/ocanaChris2/HERMES.git'
DRIVE_OUT = '/content/drive/MyDrive/HERMES/hermes_output'

# 1. Get the project
PROJ = '/content/HERMES'
if not os.path.isfile(os.path.join(PROJ, 'hermes_train.py')):
    subprocess.check_call(['git', 'clone', '--depth', '1', GIT_URL, PROJ])
os.chdir(PROJ)
if PROJ not in sys.path:
    sys.path.insert(0, PROJ)

# 2. Install deps (Colab already ships torch / numpy / matplotlib)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'constriction', 'datasets', 'huggingface_hub'])

# 3. Memory / threading flags
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['OMP_NUM_THREADS']        = '1'
import torch
if torch.cuda.is_available():
    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU — Runtime → Change runtime type → GPU (T4)')

# 4. (Optional) HuggingFace auth via a Colab Secret named "HF"
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF')
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
    print('HuggingFace: authenticated')
except Exception:
    print('HuggingFace: no token — public datasets only')

# 5. (Optional) Persist outputs to Drive so checkpoints survive disconnects.
#    hermes_train.py writes to <project>/hermes_output; symlink it into Drive.
if USE_DRIVE:
    try:
        from google.colab import drive
        if not os.path.ismount('/content/drive'):
            drive.mount('/content/drive')
        os.makedirs(DRIVE_OUT, exist_ok=True)
        link = os.path.join(PROJ, 'hermes_output')
        if not os.path.islink(link) and not os.path.exists(link):
            os.symlink(DRIVE_OUT, link)
        print('Outputs -> Drive:', DRIVE_OUT)
    except Exception as e:
        print('Drive unavailable, outputs ephemeral:', e)

print('Setup complete')

## 2 · Train + benchmark loop

Reduce `p1_epochs` / `p2_epochs` for a shorter session. With `resume=True`, a
re-run skips phases that already have a checkpoint (on Drive, if mounted).

In [ ]:
from hermes_train import train_hermes

model = train_hermes(
    p1_epochs       = 10,     # Phase 1 text   — full = 15
    p2_epochs       = 15,     # Phase 2 binary — full = 25
    resume          = True,   # continue from existing checkpoints if present
    benchmark_loop  = True,   # iterative benchmark + retrain after training
    target_score    = 88.0,   # 88 = perfect, 62 = minimum pass
    max_bench_iters = None,   # None = unlimited (stops on pass / stagnation)
)

## 3 · Inspect output files

In [ ]:
import os
OUT = 'hermes_output'
for f in sorted(os.listdir(OUT)):
    fp = os.path.join(OUT, f)
    if os.path.isfile(fp):
        print(f'  {f:<45} {os.path.getsize(fp)/1024:>8.1f} KB')

## 4 · Display charts

In [ ]:
from IPython.display import Image, display
import os
OUT = 'hermes_output'
for chart in ['training_curve.png',
              'benchmark_report_iter000.png',
              'benchmark_iteration_history.png',
              'final_dashboard.png']:
    p = os.path.join(OUT, chart)
    if os.path.exists(p):
        print(f'\n-- {chart} --')
        display(Image(filename=p))

## 5 · Compress / decompress a file (roundtrip demo)

In [ ]:
from hermes_train import compress_file, decompress_file

# Demo on one of the project's own files; swap in any path you like.
src = 'hermes/model.py'
compress_file(src, 'demo.hrm', model)
decompress_file('demo.hrm', 'demo_restored.bin', model)

orig = open(src, 'rb').read()
back = open('demo_restored.bin', 'rb').read()
bpc  = os.path.getsize('demo.hrm') * 8 / len(orig)
print(f'roundtrip OK: {orig == back}   |   {len(orig)} B -> '
      f'{os.path.getsize("demo.hrm")} B  ({bpc:.3f} BPC)')

---
### Persistence & resuming

With `USE_DRIVE = True`, all checkpoints and outputs live in
`MyDrive/HERMES/hermes_output`. If Colab disconnects, reopen the notebook, run
**cell 1** then **cell 2** again — `resume=True` continues from the last saved
phase. To start clean, delete that Drive folder (or set `USE_DRIVE = False`).

### Out-of-memory?
Lower `BATCH_SIZE` / `P1_SEQ_LEN` / `P2_SEQ_LEN` in `hermes_train.py` (see the
"Reducing memory usage" section of the repo README).